This notebook shows how to use the 2D DCT-preconditioned pseudoinverse for large problems, both with/without a GPU.

In [1]:
import numpy as np
import jlinops

/home/jlindbloom/miniconda3/envs/jlinops/lib/python3.11/site-packages/cupyx/jit/_interface.py:173: FutureWarning: cupyx.jit.rawkernel is experimental. The interface can change in the future.
  cupy._util.experimental('cupyx.jit.rawkernel')


In [2]:
# Grid size
N_x = 256
N_y = 256
img_shape = (N_x, N_y)

# Build operators both as matrix and as black box
Lmat = jlinops.build_neumann2d_sparse_matrix(img_shape)
L = jlinops.Neumann2D(img_shape)


def Lpinv_factory(w, *args, **kwargs):
    return jlinops.CGWeightedNeumann2DPinvOperator(img_shape, weights=w, tol=1e-8, *args, **kwargs)


In [3]:
# Random weight vector
w = np.random.uniform(size=L.shape[0], low=1, high=1000)

# Build Pinv operator
Lpinv = Lpinv_factory(w)

# Apply to a random vector
rhs = np.random.normal(size=L.shape[0])
_ = Lpinv @ rhs

Time it?

In [4]:
%%time
for j in range(50):
    _ = Lpinv @ rhs

CPU times: user 7min 55s, sys: 1h 31s, total: 1h 8min 27s
Wall time: 43.2 s


# GPU version

In [4]:
import numpy as np
import cupy as cp

import jlinops

In [5]:
cp.ones(10)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [7]:
# Grid size
N_x = 256
N_y = 256
img_shape = (N_x, N_y)

# Build operators both as matrix and as black box
Lmat = jlinops.build_neumann2d_sparse_matrix(img_shape)
L = jlinops.Neumann2D(img_shape)


def Lpinv_factory(w, *args, **kwargs):
    return jlinops.CGWeightedNeumann2DPinvOperator(img_shape, weights=w, tol=1e-8, *args, **kwargs)


In [8]:
# Random weight vector
w = cp.random.uniform(size=L.shape[0], low=1, high=10)

# Build Pinv operator
Lpinv = Lpinv_factory(w)
Lpinv = Lpinv.to_gpu()

# Apply to a random vector
rhs = cp.random.normal(size=L.shape[0])
_ = Lpinv @ rhs

: 

In [5]:
Lpinv = Lpinv.to_gpu()

# Apply to a random vector
rhs = cp.random.normal(size=L.shape[0])
_ = Lpinv @ rhs

: 